# Mini-TP 7 · Seguridad con SAIF — gRPC

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por gRPC**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** acceso no autenticado al servicio y *man-in-the-middle* (canal sin cifrar).  
**Controles:** token en la *metadata* (interceptor) · TLS/mTLS · límite de tamaño de mensaje.  
**Elementos SAIF:** bases sólidas · coherencia de controles.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add grpcio grpcio-tools
```

## 1. Definir el servicio (proto) y compilarlo

In [ ]:
proto = '''syntax = "proto3";
package modelo;
service Modelo { rpc Predecir (Entrada) returns (Salida); }
message Entrada { repeated float features = 1; }
message Salida { int32 prediccion = 1; float confianza = 2; }
'''
open("modelo.proto", "w").write(proto)
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "grpc_tools.protoc", "-I.",
      "--python_out=.", "--grpc_python_out=.", "modelo.proto"],
      capture_output=True, text=True).returncode, "= 0 si compiló")

## 2. Control principal (GUIADO) — interceptor de autenticación + límite de tamaño

In [ ]:
import sys; sys.path.insert(0, ".")
import grpc
from concurrent import futures
import modelo_pb2 as pb, modelo_pb2_grpc as pbg

TOKEN = "Bearer tok-valido"     # en producción: un JWT verificado, desde el entorno

class AuthInterceptor(grpc.ServerInterceptor):
    def intercept_service(self, continuation, hcd):
        md = dict(hcd.invocation_metadata or [])
        if md.get("authorization") != TOKEN:
            return grpc.unary_unary_rpc_method_handler(
                lambda req, ctx: ctx.abort(grpc.StatusCode.UNAUTHENTICATED, "token inválido"))
        return continuation(hcd)

class Servicio(pbg.ModeloServicer):
    def Predecir(self, request, context):
        return pb.Salida(prediccion=int(sum(request.features) > 0), confianza=0.9)

servidor = grpc.server(
    futures.ThreadPoolExecutor(max_workers=2),
    interceptors=[AuthInterceptor()],
    options=[("grpc.max_receive_message_length", 1024 * 1024)])  # límite de 1 MB (anti-DoS)
pbg.add_ModeloServicer_to_server(Servicio(), servidor)
port = servidor.add_insecure_port("localhost:0"); servidor.start()

stub = pbg.ModeloStub(grpc.insecure_channel(f"localhost:{port}"))
try:
    stub.Predecir(pb.Entrada(features=[1.0, 2.0]))
except grpc.RpcError as e:
    print("sin token:", e.code().name)
r = stub.Predecir(pb.Entrada(features=[1.0, 2.0]), metadata=[("authorization", TOKEN)])
print("con token:", r.prediccion, round(r.confianza, 2))
servidor.stop(0)

## 3. Tu entrega — completa los `# TODO`

In [ ]:
# TODO 1: en vez de un token fijo, verifica un JWT en el interceptor
#   (jwt.decode con tu secreto) y extraé el rol; rechazá si no alcanza.
# TODO 2: cambiá el servicio para que cargue y use TU modelo entrenado
#   (adapta el mensaje Entrada a tus features).
# TODO 3 (conceptual): documenta cómo pasarías de insecure_port a TLS
#   con add_secure_port(..., grpc.ssl_server_credentials(...)).
print("Completa los TODO sobre tu propio modelo.")

## 4. Reflexión
1. ¿Qué parte del ataque frena el interceptor y qué parte frena TLS?
2. ¿Para qué sirve el límite de tamaño de mensaje?
3. ¿Qué elemento de SAIF cubre tu control principal?

*(Escribe tu respuesta aquí.)*